# Smart Waste: Training dan Evaluasi
Aktifkan GPU melalui Runtime → Change runtime type. Unggah arsip source repository ke Colab, ekstrak, lalu sesuaikan PROJECT. Dataset RealWaste berlisensi CC BY 4.0: https://archive.ics.uci.edu/dataset/908/realwaste

In [ ]:
%pip install tensorflow==2.21.0 pillow==12.0.0 scikit-learn==1.7.2 matplotlib==3.10.7 ImageHash==4.3.2

In [ ]:
import argparse, csv, hashlib, zipfile, urllib.request
from pathlib import Path
from PIL import Image, ImageOps
import imagehash
from sklearn.model_selection import train_test_split
SOURCE = 'https://archive.ics.uci.edu/static/public/908/realwaste.zip'
MAPPING = {'Food Organics':'organic','Vegetation':'organic','Paper':'paper','Plastic':'plastic'}
def prepare(root):
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    archive = root/'realwaste.zip'
    if not archive.exists(): urllib.request.urlretrieve(SOURCE, archive)
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            target = (root/info.filename).resolve()
            if not target.is_relative_to(root.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(root)
    rows, seen, hashes = [], set(), set()
    for path in sorted(root.rglob('*')):
        label = MAPPING.get(path.parent.name)
        if not label or path.suffix.lower() not in ('.jpg','.jpeg','.png'): continue
        try:
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert('RGB')
                sha = hashlib.sha256(image.tobytes()).hexdigest()
                phash = imagehash.phash(image)
                if sha in seen or any(phash-h <= 4 for h in hashes): continue
                seen.add(sha); hashes.add(phash)
            rows.append({'path':str(path.relative_to(root)), 'label':label,'sha256':sha,'source':SOURCE})
        except (OSError, ValueError): continue
    train, rest = train_test_split(rows,test_size=.30,stratify=[x['label'] for x in rows],random_state=42)
    val, test = train_test_split(rest,test_size=.50,stratify=[x['label'] for x in rest],random_state=42)
    with (root/'manifest.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=['path','label','sha256','source','split']);writer.writeheader()
        for split, data in [('train',train),('validation',val),('test',test)]:
            for row in data:writer.writerow({**row,'split':split})
    print({k:len(v) for k,v in [('train',train),('validation',val),('test',test)]})

prepare('/content/waste-data')

In [ ]:
import argparse, csv, json, hashlib
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
LABELS = ['organic','paper','plastic']
def run(root, output):
    tf.keras.utils.set_random_seed(42)
    root, output = Path(root),Path(output);output.mkdir(parents=True,exist_ok=True)
    rows=list(csv.DictReader((root/'manifest.csv').open()))
    def dataset(split, training=False):
        subset=[r for r in rows if r['split']==split]
        def generate():
            for r in subset:
                with Image.open(root/r['path']) as im:
                    im=ImageOps.exif_transpose(im).convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
                    yield np.asarray(im,dtype=np.float32), LABELS.index(r['label'])
        ds=tf.data.Dataset.from_generator(generate,output_signature=(tf.TensorSpec((224,224,3),tf.float32),tf.TensorSpec((),tf.int32)))
        if training:ds=ds.shuffle(len(subset),seed=42)
        return ds.batch(32).prefetch(tf.data.AUTOTUNE)
    inputs=tf.keras.Input((224,224,3))
    x=tf.keras.layers.RandomFlip('horizontal')(inputs)
    x=tf.keras.layers.RandomRotation(.08)(x)
    x=tf.keras.layers.Rescaling(1./127.5,offset=-1)(x)
    base=tf.keras.applications.MobileNetV2(include_top=False,weights='imagenet',input_shape=(224,224,3));base.trainable=False
    x=base(x,training=False);x=tf.keras.layers.GlobalAveragePooling2D()(x);x=tf.keras.layers.Dropout(.25)(x)
    model=tf.keras.Model(inputs,tf.keras.layers.Dense(3,activation='softmax')(x))
    y=np.array([LABELS.index(r['label']) for r in rows if r['split']=='train'])
    weights=dict(enumerate(compute_class_weight('balanced',classes=np.arange(3),y=y)))
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True),tf.keras.callbacks.ModelCheckpoint(str(output/'model.keras'),save_best_only=True)]
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    head=model.fit(dataset('train',True),validation_data=dataset('validation'),epochs=15,class_weight=weights,callbacks=callbacks)
    base.trainable=True
    for layer in base.layers[:-30]: layer.trainable=False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    fine=model.fit(dataset('train',True),validation_data=dataset('validation'),epochs=20,class_weight=weights,callbacks=callbacks)
    model=tf.keras.models.load_model(output/'model.keras')
    truth=np.array([LABELS.index(r['label']) for r in rows if r['split']=='test'])
    prediction=np.argmax(model.predict(dataset('test')),axis=1)
    accuracy=float(accuracy_score(truth,prediction))
    report=classification_report(truth,prediction,labels=[0,1,2],target_names=LABELS,output_dict=True,zero_division=0)
    (output/'evaluation.json').write_text(json.dumps({'accuracy':accuracy,'target_met':accuracy>=.8,'classification_report':report,'confusion_matrix':confusion_matrix(truth,prediction).tolist()},indent=2))
    ConfusionMatrixDisplay.from_predictions(truth,prediction,display_labels=LABELS);plt.tight_layout();plt.savefig(output/'confusion-matrix.png');plt.close()
    for metric in ['accuracy','loss']:
        plt.plot(head.history[metric]+fine.history[metric],label='training');plt.plot(head.history['val_'+metric]+fine.history['val_'+metric],label='validation');plt.legend();plt.savefig(output/(metric+'.png'));plt.close()
    checksum=hashlib.sha256((output/'model.keras').read_bytes()).hexdigest()
    (output/'metadata.json').write_text(json.dumps({'version':checksum[:12],'sha256':checksum,'labels':LABELS,'input_size':[224,224],'preprocessing':'RGB, EXIF transpose, bilinear resize; normalization embedded'},indent=2))
    print('Testing accuracy:',accuracy)

run('/content/waste-data','/content/waste-model')

In [ ]:
import json
from IPython.display import display, Image
print(json.load(open('/content/waste-model/evaluation.json')))
display(Image('/content/waste-model/confusion-matrix.png'))

In [ ]:
import shutil
shutil.copy('/content/waste-data/manifest.csv','/content/waste-model/manifest.csv')
shutil.make_archive('/content/waste-model','zip','/content/waste-model')
from google.colab import files
files.download('/content/waste-model.zip')